# 🧬** An Integrated ML-QSAR Pipeline for CDK1 Bioactivity Prediction**
### *Targeted Bioprospecting of Marine Metabolites in Breast Cancer Therapeutics*

---
**Author details:**

**Mithon Saha**

**Department of Chemistry**

**University of Chittagong**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Python](https://img.shields.io/badge/Python-3.10%2B-blue.svg)](https://www.python.org/)

## 📌 Project Overview
This repository contains the end-to-end Machine Learning-driven Quantitative Structure-Activity Relationship (ML-QSAR) pipeline developed for identifying potent inhibitors against **Cyclin-Dependent Kinase 1 (CDK1; ChEMBL ID: CHEMBL308)**.

### 🔬 Core Workflow:
1. **Data Curation:** Automated ChEMBL retrieval, salt stripping, standardization, and pIC50 conversion.
2. **Hybrid Feature Engineering:** 1024-bit Morgan Fingerprints (ECFP4) + 7 Physicochemical Descriptors via RDKit.
3. **Model Development:** Hyperparameter optimization via **Optuna** across XGBoost, LightGBM, Random Forest, and a Stacking Ensemble Regressor.
4. **Validation (OECD Compliant):** 5-fold cross-validation, external test validation, Y-randomization analysis, and Applicability Domain (Williams Plot/Leverage via PCA).
5. **Explainability & Screening:** Model interpretability using **SHAP** and high-throughput virtual screening of seaweed natural products.

---

## 💻 Environment & Reproducibility
* **Primary Platform:** Google Colab / Linux x86_64
* **Python Runtime:** Python >= 3.10

**Part 1: Data Acquisition and Preprocessing**

Target: Cyclin-dependent kinase 1

Source: ChEMBL Database

**Step 1:** Install and Import Libraries

In [ ]:
# Install ChEMBL web resource client
%%capture
!pip install chembl_webresource_client

import pandas as pd
from chembl_webresource_client.new_client import new_client
import numpy as np

**Step 2:** Determining the Target CHEMBL ID





In [ ]:
# DATA RETRIEVAL
target_id = "CHEMBL308" # Cyclin-dependent kinase 1
activity = new_client.activity
res = activity.filter(target_chembl_id=target_id) \
              .filter(standard_type="IC50") \
              .filter(standard_relation="=") \
              .filter(standard_units="nM")

df = pd.DataFrame.from_dict(res)
df = df.dropna(subset=['standard_value', 'canonical_smiles'])
df['standard_value'] = pd.to_numeric(df['standard_value'])

In [ ]:
df

**Step 3:** Handling Missing Values and Cleaning

In [ ]:
# Remove missing values for standard_value and smiles
df2 = df[df.standard_value.notna()]
df2 = df2[df2.canonical_smiles.notna()]

# Remove duplicate SMILES to ensure unique compounds
df2_nr = df2.drop_duplicates(['canonical_smiles'])

# Select relevant columns
selection = ['molecule_chembl_id', 'canonical_smiles', 'standard_value']
df3 = df2_nr[selection].copy()

# Salt Stripping (Keeping the longest SMILES string)
def salt_stripping(smiles):
    cpd = str(smiles).split('.')
    cpd_longest = max(cpd, key=len)
    return cpd_longest
df3['canonical_smiles'] = df3['canonical_smiles'].apply(salt_stripping)

# Saving Data Set
df3.to_csv('01_file.csv', index=False)

**Step 4:** Bioactivity Classification and Data Finalization


In [ ]:
# Function for Bioactivity Labling
def label_bioactivity(value):
    val = float(value)
    if val >= 10000:
        return "inactive"
    elif val <= 1000:
        return "active"
    else:
        return "intermediate"

df3['class'] = df3['standard_value'].apply(label_bioactivity)

# Reset index
df_final = df3.reset_index(drop=True)
print("Preprocessing complete.")
df_final.head()

**Step 5:** Install RDKit and Calculate Lipinski Descriptors

In [ ]:
# Install RDKit
%%capture
!pip install rdkit

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

In [ ]:
# Function to calculate Lipinski descriptors
def calculate_descriptors(smiles):
    data = []
    for elem in smiles:
        mol = Chem.MolFromSmiles(elem)
        if mol:
            data.append([
                Descriptors.MolWt(mol),
                Descriptors.MolLogP(mol),
                Lipinski.NumHDonors(mol),
                Lipinski.NumHAcceptors(mol),
                Descriptors.TPSA(mol),
                Descriptors.RingCount(mol),
                Descriptors.FpDensityMorgan2(mol)
            ])
        else:
            data.append([None]*7)

    columnNames = ["MW", "LogP", "NumHDonors", "NumHAcceptors", "TPSA", "RingCount", "FpDensity"]
    return pd.DataFrame(data, columns=columnNames)

# Calculate all 7 descriptors
df_descriptors = calculate_descriptors(df_final.canonical_smiles)

# Combine with original dataframe
df_combined = pd.concat([df_final, df_descriptors], axis=1)
df_combined.head()

**Step 6:** Normalization and convert IC50 to pIC50

In [ ]:
import numpy as np

# 01. Normalization Function
def norm_value(input_df):
    norm = []
    for i in input_df['standard_value']:
        val = float(i)
        if val > 100000000:
            val = 100000000
        norm.append(val)

    input_df['standard_value_norm'] = norm
    x = input_df.drop('standard_value', axis=1)
    return x

# 02. pIC50 Conversion Function
def pIC50(input_df):
    pIC50_list = []
    for i in input_df['standard_value_norm']:
        molar = i * (10**-9)
        pIC50_list.append(-np.log10(molar))

    input_df['pIC50'] = pIC50_list
    x = input_df.drop('standard_value_norm', axis=1)
    return x

# 03. Applying functions
df_norm = norm_value(df_combined)
df_final = pIC50(df_norm)
df_2class = df_final.copy()

df_2class.head()

**Step 7:** Final Data Consolidation and Dataset Preparation

In [ ]:
df = df_2class.copy()

# Remove rows with missing essential values
df = df.dropna(subset=["canonical_smiles", "pIC50", "class"])
# Remove duplicates to ensure each SMILES is unique
df = df.drop_duplicates(subset=["canonical_smiles"])
# Reset the index for a clean dataframe structure
df = df.reset_index(drop=True)

# Print final dataset summary
print("✅ Cleaned dataset shape:", df.shape)
print("\n📊 Class distribution:\n", df["class"].value_counts())

# Save to CSV
df.to_csv("cleaned_dataset.csv", index=False)

**Step 8:** Hybrid Feature Engineering
(Morgan Fingerprints + PhysChem Descriptors)

**Step 9:** Data Splitting and Feature Scaling

**Step 10:** Automated Hyperparameter Tuning using Optuna

**Step 11:** Stacking Ensemble Model Training

**Step 12:** Comprehensive Model
Evaluation and Metrics Calculation



In [ ]:
# 1. INSTALLATION & LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# RDKit for Cheminformatics
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, DataStructs

# Scikit-learn & ML Models
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Optuna for Optimization
!pip install optuna
import optuna

# CONFIGURATIONS & PATHS
DATA_PATH = "/content/cleaned_dataset.csv"
PRED_XLSX = "/content/Main 687 SEAWEED.xlsx"
OUT_PRED  = "/content/Best_Model_Predictions.csv"
N_BITS    = 1024


# FEATURE ENGINEERING (HYBRID DESCRIPTORS)
def smiles_to_features(smiles):
    """Generates 1024-bit Morgan Fingerprints + 7 PhysChem Descriptors."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return np.zeros(N_BITS + 7)

   # Correct way to generate Morgan Fingerprint as NumPy array
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=N_BITS)
    fp_arr = np.zeros((N_BITS,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)

    # RDKit Physicochemical Descriptors
    desc = np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.RingCount(mol),
        Descriptors.FpDensityMorgan2(mol)
    ])
    return np.concatenate([fp_arr, desc])

# DATA LOADING & PREPARATION
df = pd.read_csv(DATA_PATH)
# Convert all SMILES to hybrid feature vectors
X = np.array([smiles_to_features(s) for s in df["canonical_smiles"]])
y = df["pIC50"].values

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train[:, -7:] = scaler.fit_transform(X_train[:, -7:])
X_test[:, -7:] = scaler.transform(X_test[:, -7:])

# HYPERPARAMETER TUNING (OPTUNA) - XGBoost & LightGBM
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": 0
    }
    model = XGBRegressor(**params)
    return cross_val_score(model, X_train, y_train, cv=5, scoring="r2").mean()

def objective_lgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "num_leaves": trial.suggest_int("num_leaves", 20, 80),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1
    }
    model = LGBMRegressor(**params)
    return cross_val_score(model, X_train, y_train, cv=5, scoring="r2").mean()



# Run optimization studies
!pip install tqdm
print("Optimizing XGBoost...")
study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=25, show_progress_bar=True)

print("\nOptimizing LightGBM...")
study_lgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=25, show_progress_bar=True)


best_xgb = XGBRegressor(**study_xgb.best_params, random_state=42, n_jobs=-1, verbosity= 0)
best_lgb = LGBMRegressor(**study_lgb.best_params, random_state=42, n_jobs=-1, verbosity= -1)
rf = RandomForestRegressor(n_estimators=500, max_depth=6, n_jobs=-1, random_state=42)

# Combining best models into a Stacking Regressor
stack = StackingRegressor(
    estimators=[("xgb", best_xgb), ("lgb", best_lgb), ("rf", rf)],
    final_estimator= LGBMRegressor(n_estimators=200,random_state=42, verbosity=-1, n_jobs=-1),
    cv=5, n_jobs=-1)

print("⏳ Calculating individual Q2_cv scores...")
xgb_cv = cross_val_score(best_xgb, X_train, y_train, cv=5, scoring="r2", n_jobs=-1).mean()
lgb_cv = cross_val_score(best_lgb, X_train, y_train, cv=5, scoring="r2", n_jobs=-1).mean()
rf_cv = cross_val_score(rf, X_train, y_train, cv=5, scoring="r2", n_jobs=-1).mean()
stack_cv = cross_val_score(stack, X_train, y_train, cv=5, scoring="r2", n_jobs=-1).mean()

#  For all model fitting
best_xgb.fit(X_train, y_train)
best_lgb.fit(X_train, y_train)
rf.fit(X_train, y_train)
stack.fit(X_train, y_train)

# MODEL EVALUATION
def evaluate_model(model, name, Q2_cv):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Error Metrics
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    # R2 and Gap calculation
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    Train_CV_Gap = train_r2 - Q2_cv  # The Gap (Train R2 - Q2_cv)

    return {
        "Model": name,
        "Q2_cv": round(Q2_cv, 4),
        "Train R²": round(train_r2,4),
        "Train_CV_Gap": round(Train_CV_Gap, 4),
        "Test R²": round(test_r2,4),
        "Train MSE": round(train_mse,4),
        "Train RMSE": round(train_rmse,4),
        "Train MAE": round(train_mae,4),
        "Test MSE": round(test_mse,4),
        "Test RMSE": round(test_rmse,4),
        "Test MAE": round(test_mae,4)
    }

Q2_cv_list = [xgb_cv, lgb_cv, rf_cv, stack_cv]
models = [best_xgb, best_lgb, rf, stack]
names = ["XGBoost", "LightGBM", "RandomForest", "Stacking"]

results = [evaluate_model(m, n, q) for m, n, q in zip(models, names, Q2_cv_list)]

# Comparison Table
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="Q2_cv", ascending=False).reset_index(drop=True)

print("\n📊 FINAL COMPARISON TABLE:")
print(df_results.to_string(index=False))


# 3. BEST MODEL SELECTION
stable_models = df_results[df_results['Train_CV_Gap'] <= 0.3]
best_row = stable_models.iloc[0] if not stable_models.empty else df_results.sort_values(by="Train_CV_Gap").iloc[0]
print(f"\n🎯 Best Model: {best_row['Model']} (Train_CV_Gap: {best_row['Train_CV_Gap']})")


# Locking final model
final_model = [m for m, n in zip(models, names) if n == best_row['Model']][0]

**Step 13:** Comprehensive Model Evaluation and Performance Comparison

**Step 14:** Data Visualization and Result Reporting (Heatmap Analysis)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# MODEL EVALUATION & DATA STORAGE
# Ensure Q2_cv_list is the list of CV scores calculated previously
results = []

# FIX: Added 'q2_val' to the unpacking and used the correct list 'Q2_cv_list'
for model, name, q2_val in zip(models, names, Q2_cv_list):
    # Predictions for both sets
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Calculate individual metrics
    tr_r2 = r2_score(y_train, y_train_pred)
    te_r2 = r2_score(y_test, y_test_pred)
    tr_mse = mean_squared_error(y_train, y_train_pred)
    te_mse = mean_squared_error(y_test, y_test_pred)
    tr_mae = mean_absolute_error(y_train, y_train_pred)
    te_mae = mean_absolute_error(y_test, y_test_pred)
    gap = tr_r2 - q2_val

    # Append to results
    results.append({
        "Model": name,
        "Q2_cv": round(q2_val, 4),
        "Train R²": round(tr_r2, 4),
        "Train_CV_Gap": round(gap, 4),
        "Test R²": round(te_r2, 4),
        "Train RMSE": round(np.sqrt(tr_mse), 4),
        "Test RMSE": round(np.sqrt(te_mse), 4),
        "Train MAE": round(tr_mae, 4),
        "Test MAE": round(te_mae, 4),
        "Train MSE": round(tr_mse, 4),
        "Test MSE": round(te_mse, 4)
    })

# Convert to DataFrame and Sort by Q2_cv
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="Q2_cv", ascending=False).reset_index(drop=True)

# PROFESSIONAL HEATMAP VISUALIZATION
# Set "Model" as index for the heatmap
df_heatmap = df_results.set_index("Model")

# Create Plot - Smaller size for compact boxes (Width: 11, Height: 5)
plt.figure(figsize=(11, 5))

sns.heatmap(
    df_heatmap,
    annot=True,               # Show values inside cells
    fmt=".3f",                # Decimal formatting
    cmap="RdYlBu",            # Changed color palette (Professional Diverging)
    linewidths=1.5,           # Spacing between cells
    linecolor='white',
    cbar_kws={'label': 'Metric Value'},
    annot_kws={"weight": "bold", "size": 10} # Bold values inside
)

# Formatting labels with BOLD text
plt.xticks(rotation=45, ha="right", fontsize=10, fontweight='bold')
plt.yticks(rotation=0, fontsize=10, fontweight='bold')
plt.xlabel("Evaluation metrics", fontsize=12, fontweight='bold')
plt.ylabel("ML models", fontsize=12, fontweight='bold')

plt.tight_layout()

# Save the Heatmap in High Resolution
heatmap_save_path = "/content/model_heatmap_professional.png"
plt.savefig(heatmap_save_path, dpi=600, bbox_inches='tight')

# Show Plot
plt.show()

**Step 15:** Descriptor-Target Correlation Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Build a dataframe for correlation
# Combine descriptors + target into a single matrix
# (Morgan fingerprints are binary; correlation not meaningful — we usually skip them)
desc_names = ["MolWt", "MolLogP", "NumHDonor", "NumHAccept", "TPSA", "RingCount", "FpDensity"]
data_for_corr = pd.DataFrame(X[:, -7:], columns=desc_names)
data_for_corr["pIC50"] = y

# Compute correlation
corr_matrix = data_for_corr.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)

# Add text annotations
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center", va="center", color="black")

# Configure ticks
ax.set_xticks(np.arange(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(len(corr_matrix.index)))
ax.set_yticklabels(corr_matrix.index)

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Correlation", rotation=270, labelpad=15)

plt.title("Descriptor–pIC50 Correlation Heatmap")
plt.tight_layout()
plt.show()

**Step 16**: Automated Model Selection and Finalization

**Step 17:** Model Performance Visualization (Predicted vs. Experimental pIC50 Plot)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score
import numpy as np

# 1. GENERATE PREDICTIONS
# 'final_model' is the locked model chosen based on the lowest gap
y_pred_train = final_model.predict(X_train)
y_pred_test = final_model.predict(X_test)

# 2. CALCULATE PERFORMANCE METRICS
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

# FIX: Define best_model_name if it was missing to avoid NameError
try:
    model_display_name = best_model_name
except NameError:
    model_display_name = final_model.__class__.__name__

# 3. INITIALIZE PLOT AESTHETICS
plt.figure(figsize=(8, 8))
sns.set_style("white")

# Plot Training Data (Blue)
plt.scatter(y_train, y_pred_train, color='#3498db', alpha=0.6, s=80,
            label=f'Train (R²={r2_train:.3f})', edgecolors='white', linewidths=0.5)

# Plot Test Data (Red) - Added for a complete scientific plot
plt.scatter(y_test, y_pred_test, color='#e74c3c', alpha=0.6, s=80,
            label=f'Test (R²={r2_test:.3f})', edgecolors='white', linewidths=0.5)

# 4. GENERATE THE IDEAL FIT LINE (y=x)
all_values = np.concatenate([y_train, y_test, y_pred_train, y_pred_test])
axis_min, axis_max = np.floor(all_values.min()) - 0.5, np.ceil(all_values.max()) + 0.5

plt.plot([axis_min, axis_max], [axis_min, axis_max], color='black',
         linestyle='--', linewidth=1.5, label='Ideal Fit (y=x)')

# 5. SET AXIS LABELS AND TITLE (BOLD)
plt.xlabel('Experimental pIC50', fontsize=14, fontweight='bold')
plt.ylabel('Predicted pIC50', fontsize=14, fontweight='bold')
plt.title(f'Regression Analysis: {model_display_name}', fontsize=16, fontweight='bold', pad=15)

# 6. CONFIGURE LEGEND AND GRID
plt.legend(loc='upper left', fontsize=11, frameon=True, shadow=True)
plt.grid(True, linestyle=':', alpha=0.6)

# Ensure the plot is square
plt.xlim(axis_min, axis_max)
plt.ylim(axis_min, axis_max)

# 7. SAVE AND DISPLAY
plt.tight_layout()

# Save the plot with high resolution
plot_save_path = f"/content/parity_plot_{model_display_name.lower()}.png"
plt.savefig(plot_save_path, dpi=300, bbox_inches='tight')

print(f"✅ Regression plot for {model_display_name} saved successfully to: {plot_save_path}")
plt.show()

**Step 18:** Test-set validation and External virtual screening / external prediction

**Step 19:** Residual Analysis and Error Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. CALCULATE RESIDUALS (ERROR) FOR THE BEST MODEL
# Residuals = Experimental Value - Predicted Value
predictions = final_model.predict(X_test)
residuals = y_test - predictions

# FIX: Safety check for best_model_name
try:
    model_display_name = best_model_name
except NameError:
    model_display_name = final_model.__class__.__name__

# 2. INITIALIZE PLOT AESTHETICS
plt.figure(figsize=(9, 6))
sns.set_style("whitegrid") # Clean grid for standard scientific view

# 3. CREATE HISTOGRAM WITH KERNEL DENSITY ESTIMATE (KDE)
sns.histplot(residuals, kde=True, color='#0077b6', bins=25,
             edgecolor='black', alpha=0.6)

# 5. CONFIGURE LABELS AND TITLES (Publication Standard)
plt.xlabel('Residual', fontsize=14, fontweight='bold')
plt.ylabel('Frequency (Count)', fontsize=14, fontweight='bold')

# Configure Legend
plt.legend(fontsize=11, frameon=True)

# 6. SAVE AND DISPLAY THE PLOT
plt.tight_layout()
save_path_residuals = f"/content/residual_plot_{model_display_name.lower()}.png"
plt.savefig(save_path_residuals, dpi=300, bbox_inches='tight')

print(f"✅ Residual plot for {model_display_name} saved successfully to: {save_path_residuals}")
plt.show()

**Step 20:** External Lead Discovery

In [ ]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import joblib

# Ensure openpyxl is available for Excel reading
!pip install openpyxl -q

try:
    # --- AUTO-FIX FOR MISSING NAME ---
    try:
        display_name = best_model_name
    except NameError:
        # If the variable is missing, get the name from the model class
        display_name = final_model.__class__.__name__

    print(f"\n🚀 Starting External Prediction on: {PRED_XLSX}")
    print(f"Using Model: {display_name}")

    # 1. LOAD EXTERNAL DATASET
    seaweed_df = pd.read_excel(PRED_XLSX)

    # 2. VALIDATE SMILES COLUMN
    target_col = [col for col in seaweed_df.columns if col.upper() == 'SMILES']
    if not target_col:
        raise ValueError("Error: No column named 'SMILES' found in the Excel file!")

    smiles_column = target_col[0]
    print(f"✅ Found SMILES column: {smiles_column}")

    # 3. FEATURE EXTRACTION
    print("⏳ Extracting hybrid features (Morgan FP + PhysChem) from SMILES...")
    X_sea_raw = []
    invalid_indices = []

    for i, s in enumerate(tqdm(seaweed_df[smiles_column], desc="Processing SMILES")):
        # Convert SMILES to features using the function defined earlier
        features = smiles_to_features(str(s).strip())

        # Check if features are valid (not all zeros)
        if np.all(features == 0):
            invalid_indices.append(i)
        X_sea_raw.append(features)

    X_sea_raw = np.array(X_sea_raw)

    # 4. SCALING
    # Apply the same scaler used during the training phase (last 7 columns)
    print("⚖️ Scaling physicochemical descriptors...")
    X_sea_raw[:, -7:] = scaler.transform(X_sea_raw[:, -7:])

    # 5. PREDICTION
    print(f"🔮 Predicting pIC50 values using the {display_name} model...")
    seaweed_df["Predicted_pIC50"] = final_model.predict(X_sea_raw)

    # Handle invalid SMILES rows
    if invalid_indices:
        print(f"⚠️ Warning: {len(invalid_indices)} invalid SMILES detected. Setting their predictions to NaN.")
        seaweed_df.loc[invalid_indices, "Predicted_pIC50"] = np.nan

    # 6. SAVE RESULTS
    seaweed_df.to_csv(OUT_PRED, index=False)
    print(f"✅ Prediction Successful! Results saved to: {OUT_PRED}")

    # 7. IDENTIFY LEAD CANDIDATES (TOP 5)
    print("\n🌟 Top 5 Potential Drug Candidates (Highest Predicted Activity):")
    top_5 = seaweed_df.dropna(subset=['Predicted_pIC50']).sort_values(by='Predicted_pIC50', ascending=False).head(5)

    print("-" * 60)
    display_cols = [smiles_column, 'Predicted_pIC50']
    print(top_5[display_cols].to_string(index=False))
    print("-" * 60)

except NameError as e:
    print(f"❌ Error: A required variable or model is missing. Please ensure all previous cells ran successfully. Detail: {e}")
except FileNotFoundError:
    print(f"❌ Error: The file {PRED_XLSX} was not found. Please verify the file path in Colab.")
except Exception as e:
    print(f"❌ Error occurred during prediction: {e}")

**Step 21:** Applicability Domain (AD) Calculation using Leverage

**Step 22:** Model Reliability Assessment via Williams Plot

**Step 23:** Final Lead Filtering and Exporting Reliable Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import pandas as pd
import seaborn as sns

# ==========================================
# 1. CALCULATE APPLICABILITY DOMAIN (AD)
# ==========================================
def calculate_applicability_domain(X_train, X_test, X_seaweed):
    """
    Assesses model reliability using Leverage analysis (Hat Matrix).
    """
    print("\n🔍 Step 8: Calculating Applicability Domain (AD)...")

    # PCA reduction (95% variance) to handle high-dimensional fingerprints
    pca = PCA(n_components=0.95, random_state=42)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    X_sea_pca = pca.transform(X_seaweed)

    # Hat Matrix Calculation (Pseudo-Inverse for stability)
    # Formula: h = x_i * (X^T * X)^-1 * x_i^T
    xtx_inv = np.linalg.pinv(np.dot(X_train_pca.T, X_train_pca))

    def get_leverage(data_pca):
        leverages = []
        for i in range(len(data_pca)):
            h = np.dot(np.dot(data_pca[i], xtx_inv), data_pca[i].T)
            leverages.append(h)
        return np.array(leverages)

    leverage_train = get_leverage(X_train_pca)
    leverage_test = get_leverage(X_test_pca)
    leverage_sea = get_leverage(X_sea_pca)

    # Warning Leverage (h*) calculation
    # h* = 3(p+1)/n where p = features, n = training samples
    p = X_train_pca.shape[1]
    n = X_train_pca.shape[0]
    h_star = (3 * (p + 1)) / n

    return leverage_train, leverage_test, leverage_sea, h_star

# Execute AD Calculation
leverage_train, leverage_test, leverage_sea, h_star = calculate_applicability_domain(X_train, X_test, X_sea_raw)

# Assign AD Status to the Seaweed Dataset
seaweed_df['Leverage'] = leverage_sea
seaweed_df['AD_Status'] = ["Inside AD" if h <= h_star else "Outside AD (Unreliable)" for h in leverage_sea]

# ==========================================
# 2. WILLIAMS PLOT VISUALIZATION
# ==========================================
def plot_williams(y_true, y_pred, leverage, h_star, title="Williams Plot"):
    # Standardized Residuals calculation
    residuals = y_true - y_pred
    std_residuals = residuals / np.std(residuals)

    plt.figure(figsize=(10, 6))
    sns.set_style("whitegrid")

    # Plotting Test Data
    plt.scatter(leverage, std_residuals, c='#3498db', alpha=0.7, edgecolors='k', s=60, label='Test Data')

    # Threshold lines (Publication Standard)
    plt.axhline(y=3, color='r', linestyle='--', linewidth=1.5, label='Outlier Limit (±3)')
    plt.axhline(y=-3, color='r', linestyle='--', linewidth=1.5)
    plt.axvline(x=h_star, color='g', linestyle='--', linewidth=1.5, label=f'Warning Leverage (h*={h_star:.3f})')

    # Formatting (Bold Axes as requested)
    plt.title(title, fontsize=15, fontweight='bold', pad=15)
    plt.xlabel('Leverage (h)', fontsize=12, fontweight='bold')
    plt.ylabel('Standardized Residuals', fontsize=12, fontweight='bold')
    plt.xticks(fontweight='bold')
    plt.yticks(fontweight='bold')
    plt.legend(frameon=True, shadow=True, loc='best')

    plt.tight_layout()
    plt.savefig("/content/williams_plot_scientific.png", dpi=300)
    plt.show()

# Generate Williams Plot for Test Set
y_test_pred = final_model.predict(X_test)
plot_williams(y_test, y_test_pred, leverage_test, h_star, "Williams Plot: Model Reliability Domain")

# ==========================================
# 3. FINAL DATA EXPORT
# ==========================================
print("\n📝 Final Summary of Reliable Lead Candidates:")
print("-" * 60)
print(seaweed_df['AD_Status'].value_counts())
print("-" * 60)

# Identify Top 5 Candidates that are BOTH Active and Reliable
# Filter: Must be "Inside AD"
reliable_leads = seaweed_df[seaweed_df['AD_Status'] == "Inside AD"].sort_values(by='Predicted_pIC50', ascending=False).head(5)

# Save the complete analysis to Excel
FINAL_OUT = "/content/Seaweed_Lead_Compound_Analysis.xlsx"
seaweed_df.to_excel(FINAL_OUT, index=False)

print(f"\n🚀 Full Analysis Saved: {FINAL_OUT}")
print("\n🌟 Top 5 Reliable Seaweed Candidates (Inside AD):")
print(reliable_leads[[smiles_column, 'Predicted_pIC50', 'AD_Status']].to_string(index=False))

**Step 24**: Model Robustness Check via Y-Randomization

In [ ]:
# Step 24: Model Robustness Check via Y-Randomization
from sklearn.utils import shuffle
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm

def perform_y_randomization(model, X_train, y_train, X_test, y_test, n_iterations=50):
    """
    Validates model robustness by shuffling the target variable (pIC50).
    """
    print(f"⏳ Running Y-randomization test ({n_iterations} iterations)...")

    # 01. Get Original Test R2 score
    y_pred_original = model.predict(X_test)
    original_test_r2 = r2_score(y_test, y_pred_original)

    random_test_r2_scores = []

    # 02. Perform Iterations
    for i in tqdm(range(n_iterations), desc="Randomizing Y-Target"):
        # Shuffle the training target
        y_train_shuffled = shuffle(y_train, random_state=i)

        # Fit model on scrambled data
        model.fit(X_train, y_train_shuffled)

        # Predict on unscrambled test data
        y_shuffled_pred = model.predict(X_test)
        random_test_r2_scores.append(r2_score(y_test, y_shuffled_pred))

    return original_test_r2, random_test_r2_scores

# Execute the test
orig_r2, rand_r2_list = perform_y_randomization(final_model, X_train, y_train, X_test, y_test, n_iterations=50)

# 03. Calculate cRp2 metric (Todeschini's parameter)
# Formula: cRp2 = R2 * sqrt(R2 - Rr2) | Rr2 = mean random R2
mean_random_r2 = np.mean(rand_r2_list)

if orig_r2 > mean_random_r2:
    crp2 = orig_r2 * np.sqrt(orig_r2 - mean_random_r2)
else:
    crp2 = 0

# 04. Visualization of Results (Updated to match your image color)
plt.figure(figsize=(10, 6), dpi=150) # High Resolution
sns.set_style("white")

# Plot Histogram with the specific Light Blue color and white edges
sns.histplot(rand_r2_list,
             kde=True,
             color='#bde0fe',       # This is the Light Blue color from your image
             edgecolor='white',    # White border for the bars
             linewidth=1.2,
             alpha=0.8,            # Slightly higher opacity for clarity
             label='Randomized Test R² Distribution')

# Original R2 vertical line (Red Dashed)
plt.axvline(orig_r2, color='red', linestyle='--', linewidth=2.5,
            label=f'Original Test R² ({orig_r2:.3f})')

# Formatting Labels (Bold and clean)
plt.title(f"Y-Randomization Analysis (cRp2 = {crp2:.3f})", fontsize=16, fontweight='bold', pad=15)
plt.xlabel("Test R² Score", fontsize=14, fontweight='normal')
plt.ylabel("Frequency", fontsize=14, fontweight='normal')

# Bold ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Shadowed Legend (like in the image)
plt.legend(loc='lower center', frameon=True, shadow=True, fontsize=12)

# Soft grid (like in the image)
plt.grid(True, linestyle='-', alpha=0.15)

# Save the plot
plt.tight_layout()
plt.savefig("/content/y_randomization_blue_plot.png", dpi=300, bbox_inches='tight')
plt.show()

**Step 25:** SHAP Analysis for Feature Importance

In [ ]:
final_model.fit(X_train, y_train)

In [ ]:
# Step 25: SHAP Analysis for Feature Importance
!pip install shap -q
import shap
import matplotlib.pyplot as plt

# 01. Combine Feature Names (1024 Bits + 7 PhysChem)
feature_names = [f"Bit_{i}" for i in range(N_BITS)] + [
    "MolWt", "LogP", "NumHDonors", "NumHAcceptors", "TPSA", "RingCount", "MorganDensity"
]

# 02. Initialize SHAP Explainer
print("\n⏳ Step 9: Initializing SHAP Explainer (This may take a moment)...")

# Logic to handle StackingRegressor vs Single Models
if "Stacking" in str(type(final_model)):
    # For Stacking, we use the general Explainer on the predict function
    # We use a small subset of X_test (e.g., 100 samples) to speed up calculation
    explainer = shap.Explainer(final_model.predict, shap.sample(X_test, 100))
    shap_values = explainer(X_test)
else:
    # For XGBoost, LightGBM, or RF, TreeExplainer is faster and more accurate
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test)

# 03. SHAP Summary Plot (Beeswarm/Dot Plot)
# Shows the MAGNITUDE and DIRECTION of feature impact
print("📊 Generating SHAP Summary Plot...")
plt.figure(figsize=(12, 10))
if isinstance(shap_values, shap._explanation.Explanation):
    # For new Explainer API
    shap.plots.beeswarm(shap_values, max_display=20, show=False)
else:
    # For traditional TreeExplainer output
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, max_display=20, show=False)

plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=12, fontweight='bold')
plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold')

plt.tight_layout()
plt.savefig("/content/shap_summary_beeswarm.png", dpi=300, bbox_inches='tight')
plt.show()

# 04. SHAP Global Importance (Bar Plot)
# Shows the overall MAGNITUDE of feature contribution
print("📊 Generating SHAP Bar Plot...")
plt.figure(figsize=(12, 10))

if isinstance(shap_values, shap._explanation.Explanation):
    shap.plots.bar(shap_values, max_display=20, show=False)
else:
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_type="bar", max_display=20, show=False)

plt.xlabel("Mean |SHAP Value| (Average impact)", fontsize=12, fontweight='bold')
plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold')

plt.tight_layout()
plt.savefig("/content/shap_global_bar_plot.png", dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ SHAP Analysis completed. Plots saved to /content/")

**Step 26:** Download zip file

In [ ]:
import os
from google.colab import files

# =================================================================
# 1. DEFINE THE ZIP FILENAME
# =================================================================
zip_filename = "QSAR_Full_Project_Results.zip"

# =================================================================
# 2. ARCHIVE ALL RESULT FILES (CSV, PNG, PDF, PKL, XLSX)
# The -r flag includes all files matching the extensions in the current directory
# =================================================================
print("📦 Archiving all research results including plots, data, and PDFs...")
!zip -r {zip_filename} *.csv *.png *.pdf *.pkl *.xlsx

# =================================================================
# 3. DOWNLOAD THE ZIP ARCHIVE TO YOUR LOCAL MACHINE
# =================================================================
if os.path.exists(zip_filename):
    print(f"✅ ZIP archive created successfully: {zip_filename}")
    files.download(zip_filename)
else:
    print("❌ Error: No matching files were found to create the archive!")